# Mamba vs. Transformer Porównanie zdolności klasyfikacji na zbiorze danych IMDb

## 1. Setup i biblioteki

Możemy to pominąć i wykonać komendę `uv sync` jeśli notebook uruchamiamy lokalnie

### 1.1 Instalacja bibliotek

#### 1.1.1 Konkretna wersja pytorch

In [ ]:
%pip install --force-reinstall torch==2.5.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


#### 1.1.2 Pobranie skombilowanej mamby


In [ ]:
import sys
import torch

py_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])

print(f"Detected Python: {py_ver}")
print(f"Detected PyTorch: torch{torch_ver}")

base_conv = f"causal_conv1d-1.6.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"
base_mamba = f"mamba_ssm-2.3.0+cu12torch{torch_ver}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl"

conv_url = f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/{base_conv}"
mamba_url = f"https://github.com/state-spaces/mamba/releases/download/v2.3.0/{base_mamba}"

%pip install packaging ninja --quiet

print("\nUnpacking Causal-Conv1d...")
%pip install {conv_url} --no-build-isolation

print("\nUnpacking Mamba-SSM...")
%pip install {mamba_url} --no-build-isolation


#### 1.1.3 Reszta bibiliotek

In [ ]:
%pip install transformers datasets accelerate evaluate scikit-learn

## 1.2 Monkeypatch mamby

In [3]:
# Targeted diagnostic
import pkgutil
import mamba_ssm.ops.triton as triton_pkg
import causal_conv1d

print("=== mamba_ssm.ops.triton submodules ===")
print([m.name for m in pkgutil.iter_modules(triton_pkg.__path__)])

try:
    from mamba_ssm.ops.triton.selective_state_update import selective_state_update
    print("\nselective_state_update: FOUND in triton.selective_state_update")
except Exception as e:
    print(f"\nselective_state_update: NOT FOUND — {e}")

print("\n=== causal_conv1d top-level ===")
print([x for x in dir(causal_conv1d) if not x.startswith("_")])

try:
    from causal_conv1d import causal_conv1d_update
    print("\ncausal_conv1d_update: FOUND")
except Exception as e:
    print(f"\ncausal_conv1d_update: NOT FOUND — {e}")

/home/sikora/studia/sem6/llm/projekt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== mamba_ssm.ops.triton submodules ===
['k_activations', 'layer_norm', 'layernorm_gated', 'selective_state_update', 'softplus', 'ssd_bmm', 'ssd_chunk_scan', 'ssd_chunk_state', 'ssd_combined', 'ssd_state_passing']

selective_state_update: FOUND in triton.selective_state_update

=== causal_conv1d top-level ===
['causal_conv1d_fn', 'causal_conv1d_interface', 'causal_conv1d_update', 'causal_conv1d_varlen', 'cpp_functions']

causal_conv1d_update: FOUND


In [4]:
import mamba_ssm
from mamba_ssm.ops.triton.selective_state_update import selective_state_update
from causal_conv1d import causal_conv1d_fn, causal_conv1d_update


mamba_ssm.selective_state_update = selective_state_update # type: ignore

for name, val in [
    ("selective_state_update", mamba_ssm.selective_state_update), # type: ignore
    ("selective_scan_fn",      mamba_ssm.selective_scan_fn),
    ("mamba_inner_fn",         mamba_ssm.mamba_inner_fn),
    ("causal_conv1d_fn",       causal_conv1d_fn),
    ("causal_conv1d_update",   causal_conv1d_update),
]:
    print(f"{name:30s} → {bool(val):5}  {type(val)}")

all_present = all([
    mamba_ssm.selective_state_update, # type: ignore
    mamba_ssm.selective_scan_fn,
    mamba_ssm.mamba_inner_fn,
    causal_conv1d_fn,
    causal_conv1d_update,
])
print("Fast path available:", all_present)

selective_state_update         →     1  <class 'function'>
selective_scan_fn              →     1  <class 'function'>
mamba_inner_fn                 →     1  <class 'function'>
causal_conv1d_fn               →     1  <class 'function'>
causal_conv1d_update           →     1  <class 'function'>
Fast path available: True


## 1.3 Importy

In [5]:
import torch
import torch.nn as nn
import numpy as np
import evaluate
import datasets
from transformers import (
    AutoTokenizer, AutoConfig,
    MambaPreTrainedModel, MambaModel,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [6]:
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"] # type: ignore
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"] # type: ignore
    return {"accuracy": accuracy, "f1": f1}

# 2. Wczytanie datasetu i preprocessing


In [7]:
print("Is CUDA available?", torch.cuda.is_available())

Is CUDA available? True


In [8]:
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [9]:
print("\nExample from training set:")
print(dataset['train'][0])


Example from training set:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudi

# 3. Podejście 1: Pretrenowany transformer

Użycie Trainer z biblioteki transformers do fine-tuningu modelu - uczenie całego modeleu bez mrożenia wag. 
Używamy modelu `distilbert-base-uncased` do klasyfikacji sentymentu.

## Setup DistilBERT

In [10]:
import time
from transformers import (
    TrainerCallback,
    DistilBertForSequenceClassification,
    DistilBertConfig,
)

def fineTuneBert(
    max_length=512, batch_size=32, lr=2e-5, num_epochs=5, weight_decay=0.01, model_max_embed=None
):
    tokenizer = AutoTokenizer.from_pretrained(
        "distilbert/distilbert-base-uncased"
    )
    if model_max_embed is None:
        model_max_embed = max_length
        
    def tokenize_function(examples):
        return tokenizer(
            examples["text"], truncation=True, max_length=max_length
        ) 

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(
                f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds"
            )

    config = DistilBertConfig.from_pretrained(
        "distilbert/distilbert-base-uncased",
        max_position_embeddings=model_max_embed,
        num_labels=2,
        problem_type="single_label_classification"
    )
    
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert/distilbert-base-uncased",
        config=config,
        ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        logging_dir="./logs",
        logging_steps=1, 
        report_to="tensorboard", 
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            TimingCallback()
        ],  
    )

    results = trainer.evaluate()
    print("\nDistilBERT Initial Evaluation Results:")
    print(results)

    trainer.train()

    results = trainer.evaluate()
    print("\nDistilBERT Evaluation Results:")
    print(results)

In [11]:
import torch
from torch.utils.data import DataLoader

def inferenceTimEvalBERT(seq_len):
    print(f"\n--- Benchmarking Sequence Length: {seq_len} ---")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=seq_len)
        
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    test_set = tokenized_dataset["test"]
    
    config = DistilBertConfig(
        max_position_embeddings=512 if seq_len<=512 else 1024,
        num_labels=2, # type: ignore
        problem_type="single_label_classification",
    )
    model = DistilBertForSequenceClassification(config=config)
    model.to(device) # type: ignore
    model.eval()

    batch_size = 8
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    
    dataloader = DataLoader(
        test_set,  # type: ignore
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=data_collator
    )

    print("Warming up CUDA kernels...")
    warmup_batches = 3
    with torch.inference_mode():
        for i, batch in enumerate(dataloader):
            if i >= warmup_batches:
                break
            inputs = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            _ = model(input_ids=inputs, attention_mask=mask)

    if device == "cuda":
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record() # type: ignore
        
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                _ = model(input_ids=inputs, attention_mask=mask)
                
        end_event.record() # type: ignore
        torch.cuda.synchronize()
        total_time_ms = start_event.elapsed_time(end_event)
    else:
        start_time = time.perf_counter()
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                _ = model(input_ids=inputs, attention_mask=mask)
        total_time_ms = (time.perf_counter() - start_time) * 1000

    total_samples = len(test_set)
    avg_time_per_batch_ms = total_time_ms / len(dataloader)
    samples_per_second = total_samples / (total_time_ms / 1000)

    print(f"\n[Results for Seq Len {seq_len}]")
    print(f"Total Samples Processed: {total_samples}")
    print(f"Total GPU Wall Time:     {total_time_ms / 1000:.4f} seconds")
    print(f"Avg Time per Batch:      {avg_time_per_batch_ms:.2f} ms (Batch Size: {batch_size})")
    print(f"Throughput Speed:        {samples_per_second:.2f} samples/second")

## Sekwencje

### Długość sekwencji 128

In [52]:
fineTuneBert(128,batch_size=60,lr=2e-5,weight_decay=0.01,model_max_embed=512)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 18127.34it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.692383,0,0.503760,0.347371



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6923828721046448, 'eval_accuracy': 0.50376, 'eval_f1': 0.34737050473580283}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.106409,0.300647,0.868440,0.868360
2,0.189927,0.292156,0.876920,0.876885
3,0.044610,0.322605,0.877560,0.877524
4,0.039188,0.371973,0.876480,0.876450
5,0.007978,0.402539,0.875520,0.875516


Epoch 1 completed in 70.24 seconds


...

Epoch 4 completed in 70.65 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.30it/s]


Epoch 5 completed in 71.20 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.007978,0.402539,5,0.875520,0.875516



DistilBERT Evaluation Results:
{'eval_loss': 0.4025386571884155, 'eval_accuracy': 0.87552, 'eval_f1': 0.8755163160796516}


In [58]:
inferenceTimEvalBERT(128)


--- Benchmarking Sequence Length: 128 ---
Warming up CUDA kernels...

[Results for Seq Len 128]
Total Samples Processed: 25000
Total GPU Wall Time:     22.9722 seconds
Avg Time per Batch:      7.35 ms (Batch Size: 8)
Throughput Speed:        1088.27 samples/second


### Długość sekwencji 256

In [60]:
fineTuneBert(256,model_max_embed=512,batch_size=100)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5064.67it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.695482,0,0.446400,0.422908



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6954823732376099, 'eval_accuracy': 0.4464, 'eval_f1': 0.42290827064916264}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.165036,0.236498,0.905120,0.905066
2,0.326505,0.233067,0.908200,0.908189
3,0.086708,0.251738,0.908520,0.908434
4,0.089903,0.267400,0.909760,0.909724
5,0.034107,0.282746,0.911320,0.911317


Epoch 1 completed in 144.40 seconds


...

Epoch 4 completed in 143.42 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.53it/s]


Epoch 5 completed in 145.63 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.034107,0.282746,5,0.911320,0.911317



DistilBERT Evaluation Results:
{'eval_loss': 0.2827463746070862, 'eval_accuracy': 0.91132, 'eval_f1': 0.911317490072131}


In [12]:
inferenceTimEvalBERT(256)



--- Benchmarking Sequence Length: 256 ---


Map: 100%|██████████| 50000/50000 [00:03<00:00, 14638.12 examples/s]


Warming up CUDA kernels...

[Results for Seq Len 256]
Total Samples Processed: 25000
Total GPU Wall Time:     45.9677 seconds
Avg Time per Batch:      14.71 ms (Batch Size: 8)
Throughput Speed:        543.86 samples/second


### Długość sekwencji 512

In [ ]:
fineTuneBert(512,batch_size=16,model_max_embed=512)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.696974,0,0.498720,0.333471



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6969738006591797, 'eval_accuracy': 0.49872, 'eval_f1': 0.33347132795760187}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.043056,0.218847,0.916800,0.916665
2,0.029927,0.222714,0.929080,0.929071
3,0.002444,0.284495,0.931640,0.931631
4,0.003637,0.366731,0.927040,0.926992
5,0.000929,0.357567,0.931720,0.931719


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 312.27 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000929,0.357567,5,0.931720,0.931719



DistilBERT Evaluation Results:
{'eval_loss': 0.35756716132164, 'eval_accuracy': 0.93172, 'eval_f1': 0.9317193854744693}


In [13]:
inferenceTimEvalBERT(512)


--- Benchmarking Sequence Length: 512 ---


Map: 100%|██████████| 50000/50000 [00:03<00:00, 13198.62 examples/s]


Warming up CUDA kernels...

[Results for Seq Len 512]
Total Samples Processed: 25000
Total GPU Wall Time:     90.4429 seconds
Avg Time per Batch:      28.94 ms (Batch Size: 8)
Throughput Speed:        276.42 samples/second


### Długość sekwencji 1024

In [ ]:
fineTuneBert(1024,batch_size=16,lr=1e-5,num_epochs=10,weight_decay=0.01,model_max_embed=1024)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.695970,0,0.500000,0.333333



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6959699988365173, 'eval_accuracy': 0.5, 'eval_f1': 0.33333333333333326}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.266240,0.373753,0.836280,0.834751
2,0.036761,0.358681,0.850920,0.849386
3,0.014580,0.319574,0.880600,0.880594
4,0.009429,0.395163,0.878880,0.878821
5,0.006657,0.469261,0.874920,0.874769
6,0.419148,0.489335,0.874960,0.874908


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 6 completed in 589.02 seconds


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
inferenceTimEvalBERT(1024)


--- Benchmarking Sequence Length: 1024 ---


Map: 100%|██████████| 50000/50000 [00:03<00:00, 12709.22 examples/s]


Warming up CUDA kernels...

[Results for Seq Len 1024]
Total Samples Processed: 25000
Total GPU Wall Time:     140.0985 seconds
Avg Time per Batch:      44.83 ms (Batch Size: 8)
Throughput Speed:        178.45 samples/second


# 4. Podejście 2: Pretrenowana Mamba

Używamy modelu `state-spaces/mamba-130m`, który potem fine-tunujemy do zadania klasyfikacji. Biblioteka `transformers` nie oferuje gotowego modelu do tego zadania, więc napisaliśmy własny wrapper 

## Setup mamby

In [ ]:
from transformers import GPTNeoXTokenizerFast # type: ignore

def prepare_mamba_dataset(dataset, max_length=1024):
    tokenizer = GPTNeoXTokenizerFast.from_pretrained("state-spaces/mamba-130m-hf")

    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
        )

    print("Tokenizing dataset for Mamba...")
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    if "label" in tokenized_dataset["train"].column_names:
        tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

    tokenized_dataset.set_format("torch")

    return tokenized_dataset, tokenizer

In [ ]:
import mamba_ssm

if not hasattr(mamba_ssm, 'selective_state_update'):
    from mamba_ssm.ops.selective_scan_interface import (
        selective_scan_fn,
        selective_state_update, # type: ignore
    )
    mamba_ssm.selective_state_update = selective_state_update # type: ignore
    mamba_ssm.selective_scan_fn     = selective_scan_fn

if not hasattr(mamba_ssm, 'mamba_inner_fn'):
    try:
        from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn
        mamba_ssm.mamba_inner_fn = mamba_inner_fn
    except ImportError:
        # v2 renamed / removed mamba_inner_fn; None makes transformers skip it
        mamba_ssm.mamba_inner_fn = None

print("mamba_ssm patched:",
      hasattr(mamba_ssm, 'selective_state_update'),
      hasattr(mamba_ssm, 'selective_scan_fn'),
      hasattr(mamba_ssm, 'mamba_inner_fn'))

mamba_ssm patched: True True True


In [18]:
class MambaForSequenceClassification(MambaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.backbone = MambaModel(config)  # must be 'backbone' to match checkpoint keys
        self.score = nn.Linear(config.hidden_size, self.num_labels, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, use_cache=False)
        hidden  = outputs[0]  # (B, L, H)

        if attention_mask is not None:
            seq_lens = attention_mask.int().sum(-1) - 1
            pooled   = hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]
        else:
            pooled   = hidden[:, -1, :]

        logits = self.score(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states)

In [ ]:
class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

def fineTuneMambaClassification(tokenized_dataset, collator,batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
    tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True # Set this directly during config loading
    )

    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, # Pass the fully configured config
        ignore_mismatched_sizes=True
    )
    # Explicitly configure pad token ID to match the tokenizer configuration
    model.config.pad_token_id = tokenizer.pad_token_id


    # 3. Define Standard Training Arguments
    training_args = TrainingArguments(
        output_dir="./mamba_classification_results",
        eval_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        gradient_checkpointing=False,
        weight_decay=0.01,
        logging_steps=1,
        fp16=False,  # Recommended for custom CUDA compilation layouts
        bf16=True,  # Recommended for custom CUDA compilation layouts
        report_to="tensorboard"
    )

    # 4. Fire up the regular Hugging Face Trainer engine
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=collator,
        compute_metrics=compute_metrics, # Pass your accuracy/f1 metric helper
        callbacks=[TimingCallback()] # Add TensorBoardCallback and TimingCallback
    )

    trainer.train()

In [ ]:
import torch

def inferenceTimEval(seq_len):
    print(f"\n--- Benchmarking Sequence Length: {seq_len} ---")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=seq_len)
    
    if mamba_tokenizer.pad_token is None:
        mamba_tokenizer.pad_token = mamba_tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        "state-spaces/mamba-130m-hf",
        num_labels=2,
        use_mamba_kernels=True  
    )
    model = MambaForSequenceClassification.from_pretrained(
        "state-spaces/mamba-130m-hf",
        config=config, 
        ignore_mismatched_sizes=True
    ).to(device) # type: ignore
    model.eval()

    batch_size = 8
    
    test_set = tokenized_dataset["test"]
    
    data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer, return_tensors="pt")
    
    dataloader = DataLoader(
        test_set, # type: ignore
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=data_collator
    )

    print("Warming up CUDA kernels...")
    warmup_batches = 3
    with torch.inference_mode():
        for i, batch in enumerate(dataloader):
            if i >= warmup_batches:
                break
            inputs = batch["input_ids"].to(device)
            _ = model(inputs)

    if device == "cuda":
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        start_event.record()# type: ignore
        
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                _ = model(inputs)
                
        end_event.record()# type: ignore
        torch.cuda.synchronize()
        total_time_ms = start_event.elapsed_time(end_event)
    else:
        start_time = time.perf_counter()
        with torch.inference_mode():
            for batch in dataloader:
                inputs = batch["input_ids"].to(device)
                _ = model(inputs)
        total_time_ms = (time.perf_counter() - start_time) * 1000

    total_samples = len(test_set)
    avg_time_per_batch_ms = total_time_ms / len(dataloader)
    samples_per_second = total_samples / (total_time_ms / 1000)

    print(f"\n[Results for Seq Len {seq_len}]")
    print(f"Total Samples Processed: {total_samples}")
    print(f"Total GPU Wall Time:     {total_time_ms / 1000:.4f} seconds")
    print(f"Avg Time per Batch:      {avg_time_per_batch_ms:.2f} ms (Batch Size: {batch_size})")
    print(f"Throughput Speed:        {samples_per_second:.2f} samples/second")

## Długość sekwencji 128

In [ ]:
import sys
import torch # Import torch to get its __file__ attribute


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=128)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 20231.24it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.103734,0.260964,0.893280,0.893151
2,0.012458,0.356628,0.892720,0.892567
3,0.000005,0.644657,0.895440,0.895432
4,0.000011,0.842040,0.895920,0.895919
5,0.000001,0.873584,0.895920,0.895911


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]


Epoch 1 completed in 169.40 seconds


...

Epoch 3 completed in 167.20 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.65it/s]


Epoch 4 completed in 165.89 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]


Epoch 5 completed in 166.93 seconds


In [26]:
inferenceTimEval(128)


--- Benchmarking Sequence Length: 128 ---
Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 15195.84it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warming up CUDA kernels...

[Results for Seq Len 128]
Total Samples Processed: 25000
Total GPU Wall Time:     69.7934 seconds
Avg Time per Batch:      22.33 ms (Batch Size: 8)
Throughput Speed:        358.20 samples/second


## Długość sekwencji 256

In [ ]:
import sys
import torch # Import torch to get its __file__ attribute


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=256)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 14609.67it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.123468,0.201960,0.922920,0.922892
2,0.005053,0.254438,0.924240,0.924193
3,0.000107,0.395591,0.925760,0.925757
4,0.000044,0.534691,0.927400,0.927397
5,0.000035,0.573302,0.927600,0.927599


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


Epoch 1 completed in 253.29 seconds


...

Epoch 3 completed in 253.70 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]


Epoch 4 completed in 253.79 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]


Epoch 5 completed in 255.26 seconds


In [28]:
inferenceTimEval(256)


--- Benchmarking Sequence Length: 256 ---
Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 12829.70it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warming up CUDA kernels...

[Results for Seq Len 256]
Total Samples Processed: 25000
Total GPU Wall Time:     130.9437 seconds
Avg Time per Batch:      41.90 ms (Batch Size: 8)
Throughput Speed:        190.92 samples/second


## Długość sekwencji 512

In [26]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=512)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=16)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 17198.80it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.029462,0.168756,0.940000,0.939974
2,0.019181,0.182540,0.944120,0.944112
3,0.000020,0.325445,0.942840,0.942834
4,0.000719,0.412771,0.945440,0.945440
5,0.000001,0.441448,0.944880,0.944879


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]


Epoch 1 completed in 445.90 seconds


...

Epoch 3 completed in 450.96 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Epoch 4 completed in 444.77 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


Epoch 5 completed in 446.99 seconds


In [29]:
inferenceTimEval(512)


--- Benchmarking Sequence Length: 512 ---
Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 14160.06it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warming up CUDA kernels...

[Results for Seq Len 512]
Total Samples Processed: 25000
Total GPU Wall Time:     247.5346 seconds
Avg Time per Batch:      79.21 ms (Batch Size: 8)
Throughput Speed:        101.00 samples/second


## Długość sekwencji 1024

In [12]:
import sys
import torch 
from transformers import DataCollatorWithPadding


sys.modules['__main__'].__file__ = torch.__file__ 

tokenized_dataset, mamba_tokenizer = prepare_mamba_dataset(dataset, max_length=1024)

data_collator = DataCollatorWithPadding(tokenizer=mamba_tokenizer)

fineTuneMambaClassification(tokenized_dataset,data_collator,batch_size=8)

Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 10802.12it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.158752,0.201754,0.940840,0.940798
2,0.002095,0.217714,0.948840,0.948835
3,0.000056,0.343092,0.947160,0.947154
4,0.004730,0.571780,0.945680,0.945679
5,0.000000,0.608669,0.947520,0.947520


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


Epoch 1 completed in 713.34 seconds


...

Epoch 3 completed in 760.00 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]


Epoch 4 completed in 760.32 seconds


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]


Epoch 5 completed in 728.29 seconds


In [30]:
inferenceTimEval(1024)


--- Benchmarking Sequence Length: 1024 ---
Tokenizing dataset for Mamba...


Loading weights: 100%|██████████| 242/242 [00:00<00:00, 18135.75it/s]
[transformers] MambaForSequenceClassification LOAD REPORT from: state-spaces/mamba-130m-hf
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warming up CUDA kernels...

[Results for Seq Len 1024]
Total Samples Processed: 25000
Total GPU Wall Time:     347.1559 seconds
Avg Time per Batch:      111.09 ms (Batch Size: 8)
Throughput Speed:        72.01 samples/second


# 5. Analiza skalowalności

Porównanie
-   **Czasu na epokę**
-   **Czasu inferencji**
-   **Końcowej jakości modelu**: Accuracy, F1


## 5.1 Jakość modeli

| Długość sekwencji | DistilBERT                | Mamba                      |
| :---------------- | :------------------------ | :------------------------- |
| 128               | ~87.6% Accuracy, 87.6% F1 | ~89.5% Accuracy, ~89.5% F1 |
| 256               | ~91.0% Accuracy, 91.0% F1 | ~92.7% Accuracy, ~92.7% F1 |
| 512               | ~93.1% Accuracy, 93.1% F1 | ~94.5% Accuracy, ~94.5% F1 |
| 1024              | ~88.0% Accuracy, 88.0% F1 | ~94.8% Accuracy, ~94.8% F1 |

## 5.2 Czas inferencji

| Model | Długość sekwencji | Całkowity czas na gpu (s) | Średni czas na Batch (ms) | Próbki na sekundę (samples/s) |
| :--- | :---: | :---: | :---: | :---: |
| **DistilBERT** | 128 | 22.9722 | 7.35 | 1088.27 |
| **DistilBERT** | 256 | 45.9677 | 14.71 | 543.86 |
| **DistilBERT** | 512 | 90.4429 | 28.94 | 276.42 |
| **DistilBERT** | 1024 | 140.0985 | 44.83 | 178.45 |
| **Mamba** | 128 | 69.7934 | 22.33 | 358.20 |
| **Mamba** | 256 | 130.9437 | 41.90 | 190.92 |
| **Mamba** | 512 | 247.5346 | 79.21 | 101.00 |
| **Mamba** | 1024 | 347.1559 | 111.09 | 72.01 |

## 5.3 Czas treningu

| Model          | Długość sekwencji | Epoch 1 (s) | Epoch 2 (s) | Epoch 3 (s) | Epoch 4 (s) | Epoch 5 (s) | Avg / Epoch  |
| :------------- | :---------------: | :---------: | :---------: | :---------: | :---------: | :---------: | :----------: |
| **DistilBERT** |        128        |    70.24    |    70.47    |    70.63    |    70.65    |    71.20    | **~70.64s**  |
| **DistilBERT** |        256        |   144.40    |   144.52    |   145.26    |   143.42    |   145.63    | **~144.65s** |
| **DistilBERT** |        512        |   317.19    |   320.88    |   320.39    |   311.89    |   312.27    | **~316.52s** |
| **DistilBERT** |       1024        |   588.33    |   587.76    |   583.76    |   584.24    |   589.98    | **~587.18s** |
| **Mamba**      |        128        |   169.40    |   167.07    |   167.20    |   165.89    |   166.93    | **~167.30s** |
| **Mamba**      |        256        |   253.29    |   254.39    |   253.70    |   253.79    |   255.26    | **~254.09s** |
| **Mamba**      |        512        |   445.90    |   448.73    |   450.96    |   444.77    |   446.99    | **~447.47s** |
| **Mamba**      |       1024        |   713.34    |   716.99    |   760.00    |   760.32    |   728.29    | **~735.79s** |

+ Dla obu architektur, przed fine-tuningiem, accuracy i f1 wynosi około 0.5 co jest spodziewane dla problemu klasyfikacji binarnej
+ Czas treningu dla `DistilBERT` zależy prawie liniowo od długości sekwencji, dla `mamby` czas ten rośnie nieliowo
+ `Mamba` trenuje się dłużej, ale skaluje się lepiej czasowo
+ Model `Mamba` jest około 2x większy niż `DistilBERT`(130M vs. 66M parametrów)
+ Dla modelu `mamba` zwiększenie długości sekwencji z 512 na 1024 przynosi niewielkie korzyści , dla modelu `DistilBERT` pogarsza wyniki
    + Jakość mamby, jako modelu rekurencyjnego, nie pogarsza się w tak jak BERT, ponieważ długości sekwencji, na której model był trenowany mogą być dowolne. Dla modelu BERT było to 512 tokenów